# 迷你分段检索引擎（简化 TF×IDF）

内存缓冲、达到阈值后 flush 为 segment、可合并 segment；查询时对各 segment 打分再累加。仅标准库，无外部依赖。

**说明**：原脚本里的 `from engine import MiniSearchEngine` 在独立 `.py` 工程中成立；在 Notebook 中类定义于上一单元格，故演示单元格不再重复 import。

In [ ]:
from collections import defaultdict, Counter
import math


class Segment:
    def __init__(self, docs):
        self.docs = docs  # doc_id -> text
        self.inverted = defaultdict(list)  # term -> [(doc_id, tf)]
        self.doc_len = {}
        self.build()

    def tokenize(self, text):
        return text.lower().split()

    def build(self):
        for doc_id, text in self.docs.items():
            terms = self.tokenize(text)
            counts = Counter(terms)
            self.doc_len[doc_id] = len(terms)

            for term, tf in counts.items():
                self.inverted[term].append((doc_id, tf))

    def search(self, query):
        scores = defaultdict(float)
        terms = self.tokenize(query)

        for term in terms:
            postings = self.inverted.get(term, [])
            df = len(postings)

            if df == 0:
                continue

            # 简化版 IDF
            idf = math.log((len(self.docs) + 1) / (df + 1)) + 1

            for doc_id, tf in postings:
                scores[doc_id] += tf * idf

        return scores


class MiniSearchEngine:
    def __init__(self, flush_threshold=3):
        self.buffer = {}
        self.segments = []
        self.flush_threshold = flush_threshold

    def add_doc(self, doc_id, text):
        self.buffer[doc_id] = text

        if len(self.buffer) >= self.flush_threshold:
            self.flush()

    def flush(self):
        if not self.buffer:
            return

        segment = Segment(self.buffer)
        self.segments.append(segment)
        self.buffer = {}

    def search(self, query, topk=10):
        total_scores = defaultdict(float)

        # 先查已经 flush 的 segment
        for segment in self.segments:
            scores = segment.search(query)
            for doc_id, score in scores.items():
                total_scores[doc_id] += score

        # 再查还在内存 buffer 里的文档
        if self.buffer:
            temp_segment = Segment(self.buffer)
            scores = temp_segment.search(query)
            for doc_id, score in scores.items():
                total_scores[doc_id] += score

        return sorted(
            total_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:topk]

    def merge_segments(self):
        merged_docs = {}

        for segment in self.segments:
            merged_docs.update(segment.docs)

        self.segments = [Segment(merged_docs)]

: 

In [2]:
engine = MiniSearchEngine(flush_threshold=2)

engine.add_doc("1", "iphone 15 pro max case")
engine.add_doc("2", "iphone charger cable")

engine.add_doc("3", "samsung phone case")
engine.add_doc("4", "xiaomi phone charger")

print("segments:", len(engine.segments))

print(engine.search("iphone case", topk=3))

engine.merge_segments()

print("segments after merge:", len(engine.segments))
print(engine.search("phone charger", topk=3))

segments: 2
[('1', 2.4054651081081646), ('3', 1.4054651081081644), ('2', 1.0)]
segments after merge: 1
[('4', 3.0216512475319814), ('3', 1.5108256237659907), ('2', 1.5108256237659907)]
